# Wordtrail — wt_extras batch run (resumable)
Reads all timing packs from `rabiul43/wordtrail-packs`,
generates `extras-<bid>.json` skip-time files, and uploads them back.

**Resumable:** if Kaggle disconnects, re-run from Cell 3 — already-processed books are skipped.

**v2:** wt_extras now also finds more spoken forms of the notice ("Libribox", "dot org") and says WHY each track got no head cut (see the SUMMARY at the end of the report, and the `??` lines).
Set `REGENERATE` in Cell 2b to rebuild every book with the new rules; set it to `False` to only process books that have no file yet.


In [27]:
# ── 1. Install dependencies ───────────────────────────────────────────────
!pip install huggingface_hub -q

In [ ]:
%%writefile wt_extras.py
#!/usr/bin/env python3
"""wt_extras.py - find the extra spoken parts (LibriVox notice, book/author line, reader credits)
in Wordtrail timing packs, using the pack's own word-match marks (q), and write skip times.

Run:   python wt_extras.py PACK_DIR OUT_DIR [--pad-a SECS] [--pad-b SECS] [--resume]
  PACK_DIR    folder with timing-*.json.gz (or *.json) files
  OUT_DIR     gets extras-<book id>.json per book, and extras_report.txt (read it by eye)
  --pad-a     seconds to trim from the start of each cut (default 0.05)
  --pad-b     seconds to trim from the end of each cut (default 0.10)
  --resume    skip books whose extras file already exists in OUT_DIR (safe to re-run after a crash)

How it decides (all from the pack, no audio, no GPU):
  q has one mark per spoken word: '0' = the word is NOT in the book text, anything else = matched.
  - HEAD: everything before the first place where the book text really starts is extra.
          A spoken heading just before it ("Chapter 3 ...", "Part first ...") is KEPT.
  - TAIL: everything after the last place where the book text really ends is extra.
  - MIDDLE: a long unmatched run is cut ONLY if it contains a LibriVox marker phrase,
          so real book text that Whisper misheard is never cut.
"""
import argparse, gzip, json, os, re, sys

WIN, NEED = 20, 12          # a "real text" window: 12 matched words out of 20
MID_MIN_WORDS = 6           # shortest middle run we will consider
HEAD_MAX_S, MID_MAX_S, TAIL_MAX_S = 60.0, 90.0, 150.0
PAD_A, PAD_B = 0.05, 0.10   # seconds: start a little late, end a little early
HEADING = {'chapter', 'section', 'part', 'book', 'volume', 'canto', 'act', 'scene', 'story', 'poem',
           'preface', 'introduction', 'prologue', 'epilogue', 'prelude', 'contents'}
MARK = re.compile(r'(lib[a-z]*vox|public domain|recording|recorded by|read by|narrated by|proofread|'
                  r'end of (chapter|section|part|book|the)|for more information|to volunteer)')

def norm(w):
    return re.sub(r"[^a-z0-9']", '', w.lower())


NOTICE = re.compile(r"(lib[a-z]*[ -]*vox|libri[a-z]*[ -]*(box|fox|vocks|bocks)|public domain|for more information|to volunteer|dot org|"
                    r"(recording|reading|read|recorded|narrated|proofread)( has been| was| is)? by)")
def find_end(words, k, span):
    """index just after the first sentence end at or after word k, or None"""
    for x in range(k, min(len(words), k + span)):
        if re.search(r'[.!?]["\u201d\u2019\']*$', words[x]): return x + 1
    return None

def head_end(words, t, m, h):
    """End of the spoken notice at the start, or None when there is no notice. Markers are looked for up to 25 words
    past h (h = where the text match is dense). After the last marker's sentence, short sentences that are
    mostly NOT in the book text (title/author line, reader credit) go too. A sentence that starts with a
    heading word, or is long, or is mostly matched, is book text and stays."""
    lim = min(len(words), h + 25)
    flat, pos = '', []
    for x in range(lim):
        pos.append(len(flat)); flat += re.sub(r"[^a-z0-9' -]", '', words[x].lower()) + ' '
    last = None
    for mt in NOTICE.finditer(flat):
        x = max(i for i in range(lim) if pos[i] <= mt.start())
        if t[x] <= HEAD_MAX_S: last = x
    if last is None: return None
    j = find_end(words, last, 10) or last + 1
    for _ in range(3):
        e = find_end(words, j, 15)
        if e is None or e - j > 14 or j >= len(words) or norm(words[j]) in HEADING: break
        if sum(m[j:e]) / float(e - j) >= 0.35: break
        j = e
    return j

def keep_title(words):
    """Index where the extra part starts, when the track begins with a title that must stay:
       - a heading word first: "Chapter 3 of Dracula ..." / "Canto 5, Bickerment. From ..." keeps the heading part
       - "<Title> from <Book> by <Author>" (poems) keeps <Title>
       Returns 0 when there is no such title."""
    lim = min(len(words), 12)
    if norm(words[0]) in HEADING:
        for x in range(1, lim):
            w = norm(words[x])
            if w in ('from', 'by', 'of', 'in') or re.search(r'[.!?]$', words[x - 1]): return x
        return 0
    for x in range(1, lim):
        if norm(words[x]) == 'from' and not NOTICE.search(' '.join(words[:x]).lower()): return x
    return 0

def head_doubt(words, t, i0, j):
    """Why a head cut cannot be trusted, or '' when it looks fine. Spoken words go at roughly 2 to 3 per second, so:
       - a long cut that removes very few words means the pack's times are off (silence, music or a bad match)
       - a long 'title' kept before the cut means the same
       - notice words right AFTER the cut mean the cut stops too early and part of the notice would still play
    A doubtful cut is dropped: no evidence, no skip."""
    start = t[i0] if i0 > 0 else 0.0       # a cut that begins at the top of the track really removes everything from 0:00
    nw, dur = j - i0, t[j] - start
    if i0 == 0 and t[0] > 10.0:
        return 'the first transcribed word comes at %.0f s, so the pack missed the opening audio' % t[0]
    if nw > 0 and dur >= 15.0 and dur / nw > 1.5:
        return 'too slow: %d words over %.0f s' % (nw, dur)
    if sum(1 for x in range(max(0, j - 3), min(len(t), j + 3)) if t[x] == t[j]) >= 3:
        return 'several words share one time around the end of the cut, so the end time cannot be trusted'
    if i0 > 0 and t[i0] > 20.0:
        return 'the kept title before the cut lasts %.0f s' % t[i0]
    after = ' '.join(re.sub(r"[^a-z0-9' -]", '', w.lower()) for w in words[j:j + 12])
    if NOTICE.search(after):
        return 'notice words continue after the cut: ' + ' '.join(words[j:j + 12])
    return ''

def analyse(tr, pad_a=PAD_A, pad_b=PAD_B):
    words = tr['w'].split(' ')
    t = [float(x) for x in tr['t']]
    q = tr.get('q') or ''
    n = len(words)
    if n < 40 or len(t) != n or len(q) != n:
        why = 'no-q' if not q else 'bad-length'
        return {'cuts': [], 'why': why, 'hwhy': why, 'hnote': ''}
    m = [1 if c != '0' else 0 for c in q]
    if sum(m) < 0.3 * n:
        return {'cuts': [], 'why': 'little-text-match', 'hwhy': 'little-text-match', 'hnote': ''}
    pre = [0]
    for v in m: pre.append(pre[-1] + v)
    dens = lambda a, b: pre[min(n, b)] - pre[max(0, a)]
    cuts = []
    hwhy, hnote = 'cut', ''
    dropped_j = 0                             # end of a head cut the safety check threw away: nothing before it may be cut again
    # HEAD: h = first matched word that starts a dense window (an upper limit, not the cut itself)
    h = next((i for i in range(0, n - WIN) if m[i] and dens(i, i + WIN) >= NEED), None)
    if h is None:
        hwhy = 'no-text-window'               # the pack never finds the book text near the start
    else:
        j = head_end(words, t, m, h)          # only a real notice makes a head cut
        if j is None:
            hwhy = 'text-starts-at-once' if h <= 2 else 'no-notice-phrase'
            if hwhy == 'no-notice-phrase': hnote = ' '.join(words[:min(h, 25)])
        elif not (j > 0 and t[j] <= HEAD_MAX_S):
            hwhy = 'notice-too-late'; hnote = ' '.join(words[:min(j, 25)])
        if j is not None and j > 0 and t[j] <= HEAD_MAX_S:
            p = keep_title(words)             # a chapter title said BEFORE the notice is kept
            a0 = 0.0 if p == 0 or p >= j else round(t[p] - pad_a, 2)
            i0 = p if a0 else 0
            bad = head_doubt(words, t, i0, j)  # SAFETY: a cut that does not look right is dropped, the chapter plays from the start
            if bad:
                hwhy, hnote = 'doubtful-head', bad
                dropped_j = j
            else:
                cuts.append({'type': 'head', 'a': a0, 'b': round(max(0.0, t[j] - pad_b), 2), 'i': i0, 'j': j})
    # TAIL: last matched word that ends a dense window
    e = next((i for i in range(n - 1, WIN - 1, -1) if m[i] and dens(i - WIN + 1, i + 1) >= NEED), None)
    if e is not None and n - 1 - e >= 4:
        s = e + 1
        if t[-1] - t[s] <= TAIL_MAX_S:
            cuts.append({'type': 'tail', 'a': round(t[s] - pad_a, 2), 'b': None, 'i': s, 'j': n})
    # MIDDLE: marker-bearing unmatched runs between head and tail
    lo = cuts[0]['j'] if cuts and cuts[0]['type'] == 'head' else dropped_j
    hi = cuts[-1]['i'] if cuts and cuts[-1]['type'] == 'tail' else n
    i = lo
    while i < hi:
        if m[i]: i += 1; continue
        j = i
        while j < hi and (not m[j] or (j + 1 < hi and not m[j + 1] and False)): j += 1
        # let one or two stray matches inside a run stay in it
        while j < hi and j + 2 < hi and sum(m[j:j + 3]) <= 1 and not (m[j + 1] and m[j + 2]):
            j += 1
            while j < hi and not m[j]: j += 1
        if j - i >= MID_MIN_WORDS and t[j - 1] - t[i] <= MID_MAX_S:
            i0 = i + 1 if (j - i > 1 and re.search(r'[.!?]$', words[i])) else i   # run starts inside the previous sentence
            txt = ' '.join(words[i0:j]).lower()
            if MARK.search(txt):
                cuts.append({'type': 'mid', 'a': round(max(0.0, t[i0] - pad_a), 2), 'b': round(t[min(j, n - 1)] - pad_b, 2), 'i': i0, 'j': j})
        i = max(j, i + 1)
    cuts.sort(key=lambda c: c['a'])

    # MERGE: a mid cut that runs right into a tail (gap <= 2s) is redundant —
    # collapse both into a single tail cut starting at the mid's 'a'.
    merged = []
    i = 0
    while i < len(cuts):
        c = cuts[i]
        if (c['type'] == 'mid' and i + 1 < len(cuts)
                and cuts[i + 1]['type'] == 'tail'
                and cuts[i + 1]['a'] - (c['b'] or 0) <= 2.0):
            tail = cuts[i + 1]
            merged.append({'type': 'tail', 'a': c['a'], 'b': None,
                           'i': c['i'], 'j': tail['j']})
            i += 2
        else:
            merged.append(c)
            i += 1
    cuts = merged

    return {'cuts': cuts, 'why': 'ok', 'hwhy': hwhy if not any(c['type'] == 'head' for c in cuts) else 'cut', 'hnote': hnote}

def run(pack_dir, out_dir, pad_a=PAD_A, pad_b=PAD_B, resume=False):
    os.makedirs(out_dir, exist_ok=True)
    all_fns = sorted(fn for fn in os.listdir(pack_dir)
                     if (fn.endswith('.json.gz') or fn.endswith('.json')) and fn != 'index.json')
    total = len(all_fns)
    done = skipped = 0
    rep = []
    HW = {}   # why tracks got no head cut
    rep_path = os.path.join(out_dir, 'extras_report.txt')
    if not resume:
        open(rep_path, 'w').close()   # fresh run: start an empty report; with --resume new books are appended
    nrep = 0
    for fn in all_fns:
        p = os.path.join(pack_dir, fn)
        try:
            d = json.loads((gzip.open(p).read() if fn.endswith('.gz') else open(p, 'rb').read()).decode())
        except Exception as e:
            print('skip unreadable %s: %s' % (fn, e)); continue
        if not isinstance(d, dict) or not isinstance(d.get('tracks'), dict):   # e.g. index.json: not a pack
            print('skip %s: not a timing pack' % fn); continue
        bid = str(d.get('bid') or re.sub(r'\D', '', fn))
        out_path = os.path.join(out_dir, 'extras-%s.json' % bid)
        if resume and os.path.exists(out_path):
            skipped += 1
            if skipped <= 3 or skipped % 100 == 0:
                print('skip %s (%d/%d)' % (bid, skipped + done, total))
            continue
        out = {'bid': bid, 'v': 1, 'tracks': {}}
        rep.append('=== %s  %s' % (bid, d.get('title') or ''))
        for k in sorted(d['tracks'], key=lambda x: int(x)):
            tr = d['tracks'][k]
            if not isinstance(tr, dict) or 'w' not in tr: continue
            r = analyse(tr, pad_a=pad_a, pad_b=pad_b)
            words = tr['w'].split(' ')
            out['tracks'][k] = [[c['a'], c['b'], c['type']] for c in r['cuts']]
            hw = r.get('hwhy', 'cut'); HW[hw] = HW.get(hw, 0) + 1
            htag = '' if hw == 'cut' else '  [head: %s]' % hw
            if r['cuts']:
                rep.append('# track %s  %s  (%d cuts)%s' % (k, r['why'], len(r['cuts']), htag))
            else:
                flag = '' if r['why'] == 'ok' else '  << %s' % r['why']
                rep.append('# track %s  (0 cuts)%s%s' % (k, flag, htag))
            if hw in ('no-notice-phrase', 'notice-too-late', 'doubtful-head') and r.get('hnote'):
                rep.append('        ?? head cut dropped or no notice found: %s' % r['hnote'][:200])
            total_dur = float(tr['t'][-1]) if tr.get('t') else 0.0
            for c in r['cuts']:
                i, j = c['i'], c['j']
                rep.append('   %-4s %7.2f -> %-7s REMOVED: %s' % (c['type'], c['a'], c['b'], ' '.join(words[i:j])[:230]))
                if c['type'] != 'tail':
                    rep.append('        KEPT next: %s' % ' '.join(words[j:j + 9]))
                if c['type'] == 'head' and c['b'] is not None and c['b'] > 35.0:
                    rep.append('        !! WARNING: head cut is long (%.1f s), verify' % c['b'])
                if c['type'] == 'tail' and total_dur > 0 and c['a'] < 0.6 * total_dur:
                    rep.append('        !! WARNING: tail starts early (%.0f%% through track), verify' % (100.0 * c['a'] / total_dur))
        json.dump(out, open(out_path, 'w'), separators=(',', ':'))
        with open(rep_path, 'a') as rf:   # written per book, so a crash or Kaggle disconnect keeps the report so far
            rf.write('\n'.join(rep[nrep:]) + '\n')
        nrep = len(rep)
        done += 1
        if done % 50 == 0 or done + skipped == total:
            print('processed %d / %d  (skipped %d)' % (done, total - skipped, skipped))
    if HW:
        lines = ['', '=== SUMMARY: why tracks have no head cut ===']
        lines += ['  %-20s %d' % (k, v) for k, v in sorted(HW.items(), key=lambda x: -x[1])]
        print('\n'.join(lines))
        with open(rep_path, 'a') as rf: rf.write('\n'.join(lines) + '\n')
    print('done: %d processed, %d skipped, %d report lines -> %s'
          % (done, skipped, len(rep), os.path.join(out_dir, 'extras_report.txt')))

if __name__ == '__main__':
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('pack_dir')
    ap.add_argument('out_dir')
    ap.add_argument('--pad-a', type=float, default=PAD_A, metavar='SECS',
                    help='seconds to trim from cut start (default %(default)s)')
    ap.add_argument('--pad-b', type=float, default=PAD_B, metavar='SECS',
                    help='seconds to trim from cut end (default %(default)s)')
    ap.add_argument('--resume', action='store_true',
                    help='skip books whose extras file already exists in OUT_DIR')
    args = ap.parse_args()
    run(args.pack_dir, args.out_dir, pad_a=args.pad_a, pad_b=args.pad_b, resume=args.resume)


In [ ]:
# ── 2b. Choose: rebuild everything with the new rules, or only add missing books ──
REGENERATE = True   # True: redo every book (old files are kept in extras_prev/ for comparison). False: only books with no file yet.


In [29]:
# ── 3. Restore any previously completed extras files (resume support) ────
# On first run: nothing to restore, extras_out/ starts empty — that is fine.
# On re-run after a crash: this pulls back whatever was already uploaded,
# so those books get skipped and we continue from where we left off.
import os
from huggingface_hub import snapshot_download

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')  # Kaggle Add-ons > Secrets
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN or ''

DEST = 'extras_prev' if REGENERATE else 'extras_out'
os.makedirs('extras_out', exist_ok=True); os.makedirs(DEST, exist_ok=True)
try:
    snapshot_download(
        repo_id='rabiul43/wordtrail-packs',
        repo_type='dataset',
        local_dir=DEST + '/',
        allow_patterns='extras/extras-*.json',
        token=HF_TOKEN,
    )
    # Move files from extras/ subfolder up to extras_out/ root
    import shutil, pathlib
    for f in pathlib.Path(DEST + '/extras').glob('extras-*.json'):
        shutil.move(str(f), DEST + '/' + f.name)
    restored = len(list(pathlib.Path(DEST).glob('extras-*.json')))
    print(f'Restored {restored} already-completed extras files')
except Exception as e:
    print(f'Nothing to restore (first run): {e}')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 0 files: 0it [00:00, ?it/s]

Restored 1560 already-completed extras files


In [30]:
# ── 4. Download all timing packs from HuggingFace (wt_packs_v2/<id>.json.gz) ──
print('Downloading timing packs...')
snapshot_download(
    repo_id='rabiul43/wordtrail-packs',
    repo_type='dataset',
    local_dir='timing_packs/',
    allow_patterns='wt_packs_v2/*.json.gz',
    token=HF_TOKEN,
)
import glob
files = glob.glob('timing_packs/wt_packs_v2/*.json.gz')
print(f'Downloaded {len(files)} timing packs')
assert files, 'No packs downloaded: check the HF_TOKEN secret and that wt_packs_v2/ has .json.gz files'

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1560 files:   0%|          | 0/1560 [00:00<?, ?it/s]

Downloaded 1560 timing packs


In [31]:
# ── 5. Flatten all packs into one folder ─────────────────────────────────
import shutil, pathlib

flat = pathlib.Path('packs_flat')
flat.mkdir(exist_ok=True)

for src in pathlib.Path('timing_packs/wt_packs_v2').glob('*.json.gz'):
    dst = flat / src.name
    if not dst.exists():
        shutil.copy2(src, dst)

print(f'Flat pack count: {len(list(flat.iterdir()))}')

Flat pack count: 1560


In [32]:
# ── 6. Run wt_extras on all packs ────
# REGENERATE: every book is redone. Otherwise --resume skips books that already have a file.
!python wt_extras.py packs_flat/ extras_out/ {'' if REGENERATE else '--resume'}

import glob, os
out_files = glob.glob('extras_out/extras-*.json')
print(f'Total extras files now: {len(out_files)}')
print('report exists:', os.path.exists('extras_out/extras_report.txt'))


skip 10017 (1/1560)
skip 10024 (2/1560)
skip 10034 (3/1560)
skip 1110 (100/1560)
skip 11983 (200/1560)
skip 1303 (300/1560)
skip 14216 (400/1560)
skip 166 (500/1560)
skip 2027 (600/1560)
skip 2651 (700/1560)
skip 327 (800/1560)
skip 3935 (900/1560)
skip 4800 (1000/1560)
skip 573 (1100/1560)
skip 6593 (1200/1560)
skip 7513 (1300/1560)
skip 8257 (1400/1560)
skip 9325 (1500/1560)
done: 0 processed, 1560 skipped, 0 report lines -> extras_out/extras_report.txt
Total extras files now: 1560
report exists: True


In [ ]:
# ── 6b. What changed compared with the old files (only when REGENERATE) ──
import json, glob, os
if REGENERATE:
    gained = lost = changed = same = 0; ex_gain = []
    for f in sorted(glob.glob('extras_out/extras-*.json')):
        old = os.path.join('extras_prev', os.path.basename(f))
        if not os.path.exists(old): continue
        a = json.load(open(old))['tracks']; b = json.load(open(f))['tracks']
        for k in b:
            ha = [c for c in a.get(k, []) if c[2] == 'head']; hb = [c for c in b[k] if c[2] == 'head']
            if not ha and hb: gained += 1; ex_gain.append((os.path.basename(f), k, hb[0]))
            elif ha and not hb: lost += 1; print('LOST head cut:', os.path.basename(f), 'track', k, ha[0])
            elif ha and hb and abs(ha[0][1] - hb[0][1]) > 0.5: changed += 1
            else: same += 1
    print(f'tracks that GAINED a head cut: {gained}   LOST one: {lost}   head time changed >0.5s: {changed}   unchanged: {same}')
    for x in ex_gain[:15]: print('  gained:', x)
else:
    print('REGENERATE is off, nothing to compare.')


In [33]:
# ── 7. Check the report for warnings ─────────────────────────────────────
with open('extras_out/extras_report.txt') as f:
    lines = f.readlines()

warnings = [l.strip() for l in lines if '!! WARNING' in l]
skipped  = [l.strip() for l in lines if '<<' in l]

print(f'Total report lines : {len(lines)}')
print(f'Warnings           : {len(warnings)}')
print(f'Skipped tracks     : {len(skipped)}')

if warnings:
    print('\n--- WARNINGS (verify these tracks before uploading) ---')
    for w in warnings: print(' ', w)

if skipped:
    print('\n--- SKIPPED (no-q / bad-length / little-text-match) ---')
    for s in skipped[:20]: print(' ', s)
    if len(skipped) > 20:
        print(f'  ... and {len(skipped)-20} more')

Total report lines : 89881
Warnings           : 173
Skipped tracks     : 64

--- WARNINGS (verify these tracks before uploading) ---
  !! WARNING: head cut is long (44.0 s), verify
  !! WARNING: head cut is long (43.4 s), verify
  !! WARNING: head cut is long (35.6 s), verify
  !! WARNING: head cut is long (35.1 s), verify
  !! WARNING: head cut is long (58.1 s), verify
  !! WARNING: head cut is long (36.2 s), verify
  !! WARNING: head cut is long (35.2 s), verify
  !! WARNING: head cut is long (37.3 s), verify
  !! WARNING: head cut is long (43.5 s), verify
  !! WARNING: head cut is long (35.8 s), verify
  !! WARNING: head cut is long (42.3 s), verify
  !! WARNING: head cut is long (38.2 s), verify
  !! WARNING: head cut is long (36.3 s), verify
  !! WARNING: head cut is long (40.6 s), verify
  !! WARNING: head cut is long (37.9 s), verify
  !! WARNING: head cut is long (38.4 s), verify
  !! WARNING: head cut is long (37.2 s), verify
  !! WARNING: head cut is long (42.0 s), verify
  !

In [ ]:
# ── 7b. Safety check before upload: files marked BAD are held back, nothing else changes ──
# "doubtful-head" in the SUMMARY above = cuts the program threw away because they looked wrong; those chapters play from the start.
import json, glob, os, re, shutil, subprocess
def bad_reasons(path):
    d = json.load(open(path))
    if d.get('v') != 1 or not isinstance(d.get('tracks'), dict): return ['not a valid skip file']
    out = []
    for k, cuts in d['tracks'].items():
        h = [c for c in cuts if c[2] == 'head']; t = [c for c in cuts if c[2] == 'tail']
        if len(h) > 1 or len(t) > 1: out.append('track %s: two head or tail cuts' % k)
        for c in cuts:
            if c[2] in ('head', 'mid') and (c[1] is None or not c[1] > c[0]): out.append('track %s: cut ends before it starts' % k)
            if c[0] is not None and c[0] < 0: out.append('track %s: negative time' % k)
        if h and t and h[0][1] and t[0][0] is not None and t[0][0] <= h[0][1]: out.append('track %s: outro starts before the notice ends' % k)
    return out
os.makedirs('extras_held', exist_ok=True)
held = 0
for f in glob.glob('extras_out/extras-*.json'):
    r = bad_reasons(f)
    if r:
        held += 1; print('HELD BACK', os.path.basename(f), r[:2]); shutil.move(f, 'extras_held/' + os.path.basename(f))
print('files held back:', held, '| files ready to upload:', len(glob.glob('extras_out/extras-*.json')))
nohead = [os.path.basename(f) for f in glob.glob('extras_out/extras-*.json')
          if not any(c[2] == 'head' for t in json.load(open(f))['tracks'].values() for c in t)]
print('books where NO track has a notice cut (they play the full notice):', len(nohead))


In [ ]:
# ── 7c. Write the list of "skip ready" books: extras_out/index.json ──────────
# The app reads this one small file for its "Skip ready books" tab.
# Format: {"v":1,"b":{"<book id>":[chapters with a notice cut, total chapters], ...}}
# Only READY books are listed. Rule (change it here, nowhere else): MORE THAN HALF of the chapters have a head cut.
# Uses only the files in extras_out/ (files held back by 7b are not there, so they are not listed).
import json, glob, os

def is_ready(with_cut, total):
    return total > 0 and with_cut * 2 > total

ready, books_seen, bad_files = {}, 0, []
for f in sorted(glob.glob('extras_out/extras-*.json')):
    bid = os.path.basename(f)[len('extras-'):-len('.json')]   # the id the app asks for, taken from the file name
    try:
        tracks = json.load(open(f))['tracks']
        assert isinstance(tracks, dict)
    except Exception as e:
        bad_files.append((os.path.basename(f), str(e))); continue
    books_seen += 1
    total = len(tracks)
    with_cut = sum(1 for cuts in tracks.values() if any(len(c) > 2 and c[2] == 'head' for c in cuts))
    if is_ready(with_cut, total):
        ready[bid] = [with_cut, total]

with open('extras_out/index.json', 'w') as fh:
    json.dump({'v': 1, 'b': ready}, fh, separators=(',', ':'), sort_keys=True)

print('books checked:', books_seen, '| ready (more than half of chapters have a notice cut):', len(ready),
      '| not ready:', books_seen - len(ready))
for bid, (a, b) in list(ready.items())[:5]:
    print('  example: book %s  %d of %d chapters' % (bid, a, b))
if bad_files:
    print('could not read (left out of the list):', bad_files[:5])
print('index.json written: %d bytes' % os.path.getsize('extras_out/index.json'))


In [34]:
# ── 8. Upload extras files back to HuggingFace ───────────────────────────
# Only run this after you are happy with the report above.
# Safe to run after a partial run too — only new files get uploaded.
# extras_out/index.json (made by cell 7c) goes up with the rest, to extras/index.json. Only the report stays here.
import os
assert os.path.exists('extras_out/index.json'), 'Run cell 7c first: extras_out/index.json is missing'
from huggingface_hub import upload_folder

upload_folder(
    repo_id='rabiul43/wordtrail-packs',
    repo_type='dataset',
    folder_path='extras_out/',
    path_in_repo='extras/',
    ignore_patterns=['extras_report.txt'],
    token=HF_TOKEN,
    commit_message='add extras skip-time files (wt_extras batch run)',
)
print('Upload complete.')

Upload complete.


In [35]:
# ── 9. (Optional) Save the report to HuggingFace for reference ───────────
from huggingface_hub import upload_file

upload_file(
    repo_id='rabiul43/wordtrail-packs',
    repo_type='dataset',
    path_or_fileobj='extras_out/extras_report.txt',
    path_in_repo='extras/extras_report.txt',
    token=HF_TOKEN,
    commit_message='add extras_report.txt',
)
print('Report uploaded.')

Report uploaded.


In [36]:
import re, random
lines = open('extras_out/extras_report.txt').read().split('\n')
book = track = ''
items = []
for i, l in enumerate(lines):
    if l.startswith('==='): book = l[4:].strip()
    elif l.startswith('# track'): track = l.split()[2]
    elif '!! WARNING' in l:
        m = re.search(r'\((\d+\.\d) s\)', l)
        items.append((float(m.group(1)) if m else 0, book, track, lines[i-2].strip(), lines[i-1].strip()))
print(len(items), 'warnings; books affected:', len({x[1] for x in items}))
items.sort(reverse=True)
def show(x):
    print(f'\n[{x[0]:.1f}s] {x[1]}  track {x[2]}\n  {x[3][:400]}\n  {x[4][:160]}')
print('\n===== 12 LONGEST ====='); [show(x) for x in items[:12]]
print('\n===== 12 RANDOM ====='); random.seed(1); [show(x) for x in random.sample(items, min(12, len(items)))]

173 warnings; books affected: 100

===== 12 LONGEST =====

[58.1s] 10287  Bohème  track 0
  head    0.00 -> 58.12   REMOVED: Dramatus to Spersonae and Play Synopsis of La Boheme, by Giuseppe Giacosa, translated by W. Grist and others. This is a LibriVox recording. All LibriVox recordings are in the public domain. For more information or to volunteer, pl
  KEPT next: Marcel, a painter baritone colline a philosopher bass alcindoro

[57.0s] 7847  Misalliance  track 0
  head   20.95 -> 56.97   REMOVED: Johnny Tarleton by Sean Daly Bentley Summer Hayes, read by MB Hypatia Tarleton, read by Ariel Lipshaw Mrs. Tarleton, read by Bev Stevens Lord Summer Hayes, read by Bruce Pirrie John Tarleton, read by Bob Newfeld Joey Percival, rea
  KEPT next: johnny tarleton an ordinary young business man of thirty

[54.6s] 1775  track 0
  head    0.00 -> 54.63   REMOVED: to be or not to be, by Kurt Vonnegut, Jr., narrated by Dennis Sayers. The Hospital Orderly, played by Mark Smith. Painter, read by Alan D

[None, None, None, None, None, None, None, None, None, None, None, None]